# MeguvizCAC v0.5.0-ApexLoop A100-80GB 全量训练（超越 OjinjinNet 高标准版）

完整落地 `docs/roadmap-v0.5.0-research-plan.md` 与 `v0.4.0` R1/R2/R3 全部实战经验：
1. **Modern `mmBERT-small` 跨词表迁移 + `24k SuperBPE` 零 `[UNK]`**：彻底取代老旧 `bge-small-zh` 21k 词表，原生打通中/英/日/韩/代码与 `M04` 中英技术评论。
2. **端侧极速微架构（Phase-0 本地实测 `1.06 ms/row`）**：淘汰 `mHC-4` 与 `CharStem`，采用 `1L RoPE+RMSNorm+GeGLU Prelude (带 Exit-0) + 32-d 零隐私 PIFE 文体 + 静态 16-Slot 工作台 + 1L 100% INT8 Core (2.26 MB) + Coda (0.23 MB) + L2 256-d 原型库 (60 KB)`，Edge-W4 总重 `<= 6.5 MB`。
3. **四道反污染数据铁律**：保护桶白名单（剔除 `48,500` 条弱伪标污染）、非静默零泄漏审计（剔除全部 R2 陷阱撞车对，119 对零泄漏 `v05-gateguard-pairs` 50/50 分层采样）、`Gold Retention = 100%` 硬断言。
4. **四阶步数恒定闭环**：Stage-0 `400` 步 `100% RTD` → Stage-1 `MuonClip` 3 轮全库蒸馏（持久存档 `v05_stage1_ckpt.pt` 到 Drive）→ Stage-2 固定 `2,400` 步 EMA+PIFE 对抗淬炼 → Stage-3 固定 `3,000` 步 `LoopQ` 全轨迹 QAT（覆盖 `R=2/4`）+ SWA。
5. **大样本平滑校准**：`n=2,400` 独立分层验证集 + 锐利温度烘焙（`T ∈ [0.9, 1.8]`）+ 分长度桶 Smooth PAVA + `Split Conformal 3-Way`（目标灰区 `<= 18%`）。

> **操作步骤**：请先手动连接 **A100 80GB（高 RAM）** 运行时，然后依次执行 **Cell 1 → Cell 2** 即可（全程自动同步 Google Drive + 15 项门禁打分 + 自动下载）。


In [ ]:
# Cell 1: 挂载 Drive、拉取 v0.5.0 代码、自动下载并清洗冻结语料（新开机子一键就绪，若已有缓存自动秒过）
import hashlib, json, os, shutil, subprocess, time, torch
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
DRIVE_OUT = Path('/content/drive/MyDrive/MeguvizCAC/v050_apexloop')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
(DRIVE_OUT / '_session_check.txt').write_text(f'mounted_utc={time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}\n', encoding='utf-8')

if not torch.cuda.is_available():
    raise RuntimeError('未检测到 GPU！请先在右上角手动连接 A100 GPU 运行时后再执行本单元。')
print('GPU:', torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory/(1024**3):.1f} GB | bf16={torch.cuda.is_bf16_supported()}')

REPO_DIR = '/content/MeguvizCAC'
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/jichuo1/MeguvizCAC.git {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} pull --ff-only
%cd /content/MeguvizCAC
!git log -3 --oneline
!pip -q install -U tokenizers onnx onnx_ir ml_dtypes onnxruntime transformers scipy scikit-learn huggingface_hub

# 自动准备 /content/data 语料资产与 R3 人设清洗（若已存在则跳过下载）
DATA_DIR = Path('/content/data')
for d in (DATA_DIR / 'stage2', DATA_DIR / 'co-teacher-gate'):
    d.mkdir(parents=True, exist_ok=True)
REL = 'https://github.com/jichuo1/MeguvizCAC/releases/download'

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for c in iter(lambda: f.read(1 << 20), b''):
            h.update(c)
    return h.hexdigest()

def ensure_dl(url: str, dst: Path, pref: str | None = None):
    if dst.is_file() and dst.stat().st_size > 0 and (pref is None or sha256_file(dst).startswith(pref)):
        print(f'[cache hit] {dst.name}')
        return
    print(f'[download] {dst.name} ...')
    subprocess.run(['curl', '-fsSL', url, '-o', str(dst)], check=True)
    if pref:
        assert sha256_file(dst).startswith(pref), f'SHA mismatch: {dst.name}'

ensure_dl(f'{REL}/v0.2-data/stage2_distill_bundle.zip', DATA_DIR / 'stage2_distill_bundle.zip')
!unzip -qo /content/data/stage2_distill_bundle.zip -d /content/data/stage2
ensure_dl(f'{REL}/v0.2-qwen-raw/stage2_qwen_bundle.zip', DATA_DIR / 'stage2_qwen_bundle.zip')
!unzip -qoj /content/data/stage2_qwen_bundle.zip -d /content/data/stage2
for name, dst, pref in [
    ('v04-qwen38-twins.jsonl', DATA_DIR / 'v04-qwen38-twins.jsonl', '7ff4227c'),
    ('v04-inkling-wk-pairs.jsonl', DATA_DIR / 'v04-inkling-wk-pairs.jsonl', '0e877a7c'),
    ('v04_semantic_intent_cache.npz', DATA_DIR / 'v04_semantic_intent_cache.npz', '9880e38b'),
    ('v04-merged-1m4.jsonl.gz', DATA_DIR / 'v04-merged-1m4.jsonl.gz', '11b63db0'),
    ('v04-twins-clean-pairs.jsonl', DATA_DIR / 'co-teacher-gate' / 'v04-twins-clean-pairs.jsonl', '3cae814d'),
]:
    ensure_dl(f'{REL}/v0.4-data/{name}', dst, pref)

r3_corpus = DATA_DIR / 'v04-merged-1m4-r3.jsonl.gz'
r3_rep = DATA_DIR / 'r3_persona_labelfix_report.json'
if not (r3_corpus.is_file() and r3_rep.is_file()):
    subprocess.run([
        'python3', '-m', 'meguviz_cac.v04_persona_labelfix',
        '--in', str(DATA_DIR / 'v04-merged-1m4.jsonl.gz'),
        '--out', str(r3_corpus),
        '--report', str(r3_rep),
    ], check=True)
rep = json.loads(r3_rep.read_text(encoding='utf-8'))
assert rep['modified_rows'] == 196 and rep['non_train_byte_identical'] is True
print('V05_STEP1_READY | corpus_rows_fixed=196 | Drive:', DRIVE_OUT)


In [ ]:
# Cell 2: 启动后台抗断线 v0.5.0-ApexLoop A100-80GB 全量训练（自动 --probe-vram、双写 Drive、15 项门禁与自动下载）
import sys, importlib
if '/content/MeguvizCAC' not in sys.path:
    sys.path.insert(0, '/content/MeguvizCAC')
import meguviz_cac.colab_v05_launcher as lv05
importlib.reload(lv05)
lv05.run_v05_in_colab(poll_seconds=90)


In [ ]:
# Cell 3: 断线重连监控 / 手动补下载单元
import json
from pathlib import Path
from google.colab import files

done = Path('/content/v05_done.json')
if done.is_file():
    info = json.loads(done.read_text(encoding='utf-8'))
    print(json.dumps(info, ensure_ascii=False, indent=2))
    files.download(info['full_zip'])
else:
    !tail -n 30 /content/log_v05_driver.txt 2>/dev/null || true
    !tail -n 20 /content/log_v05_full.txt 2>/dev/null || true
